In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import os
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split

In [12]:
class AgeGenderCNN(nn.Module):
    def __init__(self, input_size=(128, 128)):
        super(AgeGenderCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(16),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2)
        )

        # Flatten sonrası boyutu otomatik hesapla
        dummy_input = torch.zeros(1, 1, *input_size)
        dummy_output = self.features(dummy_input)
        self.flattened_size = dummy_output.view(1, -1).shape[1]

        # Cinsiyet tahmini başlığı
        self.gender_head = nn.Sequential(
            nn.Linear(self.flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

        # Yaş tahmini başlığı
        self.age_head = nn.Sequential(
            nn.Linear(self.flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        gender_out = torch.sigmoid(self.gender_head(x))
        age_out = self.age_head(x)
        return gender_out, age_out


In [13]:
class AgeGenderDataset(Dataset):
    def __init__(self, X, y_gender, y_age):
        self.X = X.astype(np.float32) / 255.0
        self.y_gender = y_gender.astype(np.float32).reshape(-1, 1)
        self.y_age = y_age.astype(np.float32).reshape(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        x = np.expand_dims(x, axis=0)
        return torch.tensor(x), torch.tensor(self.y_gender[idx]), torch.tensor(self.y_age[idx])

In [14]:
def train_model(model, dataloader, criterion_g, criterion_a, optimizer, device):
    model.train()
    total_loss = 0
    for x, y_g, y_a in tqdm(dataloader):
        x, y_g, y_a = x.to(device), y_g.to(device), y_a.to(device)
        optimizer.zero_grad()
        pred_g, pred_a = model(x)
        loss_g = criterion_g(pred_g, y_g)
        loss_a = criterion_a(pred_a, y_a)
        loss = loss_g + loss_a
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

In [15]:
def evaluate_model(model, dataloader, criterion_g, criterion_a, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for x, y_g, y_a in dataloader:
            x, y_g, y_a = x.to(device), y_g.to(device), y_a.to(device)
            pred_g, pred_a = model(x)
            loss_g = criterion_g(pred_g, y_g)
            loss_a = criterion_a(pred_a, y_a)
            loss = loss_g + loss_a
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [16]:
def test_model(model, dataloader, device):
    model.eval()
    all_preds_g = []
    all_preds_a = []
    all_true_g = []
    all_true_a = []

    with torch.no_grad():
        for x, y_g, y_a in dataloader:
            x = x.to(device)
            pred_g, pred_a = model(x)
            all_preds_g += pred_g.cpu().numpy().flatten().tolist()
            all_preds_a += pred_a.cpu().numpy().flatten().tolist()
            all_true_g += y_g.numpy().flatten().tolist()
            all_true_a += y_a.numpy().flatten().tolist()

    pred_g_bin = [1 if p > 0.5 else 0 for p in all_preds_g]

    print("\n=== [TEST SONUÇLARI] ===")
    print("Cinsiyet - Accuracy :", accuracy_score(all_true_g, pred_g_bin))
    print("Cinsiyet - Precision:", precision_score(all_true_g, pred_g_bin, zero_division=0))
    print("Cinsiyet - Recall   :", recall_score(all_true_g, pred_g_bin, zero_division=0))
    print("Cinsiyet - F1-score :", f1_score(all_true_g, pred_g_bin, zero_division=0))

    print("Yaş - MAE           :", mean_absolute_error(all_true_a, all_preds_a))
    print("Yaş - RMSE          :", root_mean_squared_error(all_true_a, all_preds_a))

In [17]:
X_train = np.load("X_train_utkface.npy", allow_pickle=True)
y_age_train = np.load("y_age_train_utkface.npy")
y_gen_train = np.load("y_gender_train_utkface.npy")

X_test = np.load("X_test_utkface.npy", allow_pickle=True)
y_age_test = np.load("y_age_test_utkface.npy")
y_gen_test = np.load("y_gender_test_utkface.npy")
valid_mask = (y_gen_train == 0) | (y_gen_train == 1)

X_train = X_train[valid_mask]
y_gen_train = y_gen_train[valid_mask]
y_age_train = y_age_train[valid_mask]

X_train_split, X_val_split, y_gen_train_split, y_gen_val_split, y_age_train_split, y_age_val_split = train_test_split(X_train, y_gen_train, y_age_train, test_size=0.1, shuffle=False)

train_dataset = AgeGenderDataset(X_train_split, y_gen_train_split, y_age_train_split)
val_dataset   = AgeGenderDataset(X_val_split, y_gen_val_split, y_age_val_split)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AgeGenderCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion_g = nn.BCELoss()
criterion_a = nn.MSELoss()

best_model_path = "best_model_utk4.pth"
best_val_loss = float('inf')
for epoch in range(30):
    train_loss = train_model(model, train_loader, criterion_g, criterion_a, optimizer, device)
    val_loss   = evaluate_model(model, val_loader, criterion_g, criterion_a, device)

    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print("Yeni en iyi model kaydedildi.")
    

100%|██████████| 536/536 [00:25<00:00, 21.02it/s]


Epoch 1: Train Loss = 378.2614 | Val Loss = 345.3911
Yeni en iyi model kaydedildi.


100%|██████████| 536/536 [00:25<00:00, 21.16it/s]


Epoch 2: Train Loss = 297.2251 | Val Loss = 313.1297
Yeni en iyi model kaydedildi.


100%|██████████| 536/536 [00:25<00:00, 21.36it/s]


...

Epoch 28: Train Loss = 54.7558 | Val Loss = 230.7671


100%|██████████| 536/536 [00:25<00:00, 21.09it/s]


Epoch 29: Train Loss = 53.4697 | Val Loss = 207.5047


100%|██████████| 536/536 [00:25<00:00, 21.13it/s]


Epoch 30: Train Loss = 52.4533 | Val Loss = 249.9444


In [18]:
test_dataset = AgeGenderDataset(X_test, y_gen_test, y_age_test)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)
model = AgeGenderCNN().to(device)
model.load_state_dict(torch.load(best_model_path))
model.eval()
# Modeli test et ve sonuçları yazdır
test_model(model, test_dataloader, device)


=== [TEST SONUÇLARI] ===
Cinsiyet - Accuracy : 0.7721545569088618
Cinsiyet - Precision: 0.7725245316681534
Cinsiyet - Recall   : 0.7507585609016039
Cinsiyet - F1-score : 0.7614860408881072
Yaş - MAE           : 9.610222507354242
Yaş - RMSE          : 13.196134131988307


In [19]:
X_train = np.load("X_train_all_imdbwiki.npy", allow_pickle=True)
y_age_train = np.load("y_age_train_all_imdbwiki.npy")
y_gen_train = np.load("y_gender_train_all_imdbwiki.npy")

X_test = np.load("X_test_all_imdbwiki.npy", allow_pickle=True)
y_age_test = np.load("y_age_test_all_imdbwiki.npy")
y_gen_test = np.load("y_gender_test_all_imdbwiki.npy")

X_train_split, X_val_split, y_gen_train_split, y_gen_val_split, y_age_train_split, y_age_val_split = train_test_split(X_train, y_gen_train, y_age_train, test_size=0.1, shuffle=False)

train_dataset = AgeGenderDataset(X_train_split, y_gen_train_split, y_age_train_split)
val_dataset   = AgeGenderDataset(X_val_split, y_gen_val_split, y_age_val_split)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeGenderCNN(input_size=(64, 64)).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion_g = nn.BCELoss()
criterion_a = nn.MSELoss()

best_model_path = "best_model_imdbwiki4.pth"
best_val_loss = float('inf')
for epoch in range(30):
    train_loss = train_model(model, train_loader, criterion_g, criterion_a, optimizer, device)
    val_loss   = evaluate_model(model, val_loader, criterion_g, criterion_a, device)

    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print("Yeni en iyi model kaydedildi.")

100%|██████████| 11459/11459 [03:50<00:00, 49.77it/s]


Epoch 1: Train Loss = 185.4699 | Val Loss = 149.8919
Yeni en iyi model kaydedildi.


100%|██████████| 11459/11459 [03:42<00:00, 51.60it/s]


Epoch 2: Train Loss = 165.8388 | Val Loss = 144.9899
Yeni en iyi model kaydedildi.


100%|██████████| 11459/11459 [03:09<00:00, 60.35it/s]


...

Epoch 28: Train Loss = 97.3033 | Val Loss = 148.9945


100%|██████████| 11459/11459 [01:56<00:00, 98.69it/s]


Epoch 29: Train Loss = 96.4718 | Val Loss = 147.2062


100%|██████████| 11459/11459 [02:12<00:00, 86.48it/s]


Epoch 30: Train Loss = 95.6668 | Val Loss = 151.4042


In [20]:
test_dataset = AgeGenderDataset(X_test, y_gen_test, y_age_test)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)
model = AgeGenderCNN(input_size=(64, 64)).to(device)
model.load_state_dict(torch.load(best_model_path))
model.eval()
# Modeli test et ve sonuçları yazdır
test_model(model, test_dataloader, device)


=== [TEST SONUÇLARI] ===
Cinsiyet - Accuracy : 0.7409058330306035
Cinsiyet - Precision: 0.7450824585616104
Cinsiyet - Recall   : 0.8683509341998376
Cinsiyet - F1-score : 0.802007757928618
Yaş - MAE           : 8.90863297000631
Yaş - RMSE          : 11.819284086053667
